# AstroMap-AI: Physics-Informed Interactive Exoplanet Detection
### End-to-End Scientific Validation Pipeline & Benchmark Walkthrough

This notebook demonstrates the complete, reproducible pipeline of **AstroMap-AI**:
1. **NASA MAST Ingestion:** Ingesting calibrated TESS SPOC light curves.
2. **Dual-Domain Preprocessing:** Savitzky-Golay detrending and 2D Ricker Continuous Wavelet Transform (CWT) scalogram generation.
3. **PINN-TransNet Deep Learning:** Multi-scale 1D-CNN + TCN with Multi-Head Temporal Self-Attention and Mandel-Agol physics loss.
4. **Classical Fourier Grid Search:** Box-Least-Squares (BLS) periodogram search.
5. **Hybrid Consensus Engine:** Cross-modal harmonic resonance and parameter consensus.
6. **Robovetter Vetting Battery:** Odd-Even depth consistency, secondary eclipse search, and Kepler-3 stellar density verification against Gaia DR3.
7. **Automated Validation Report:** Self-contained publication-grade candidate validation report generation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch

# Ensure local src is in Python path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root_dir / "python" / "src"))

from astromap.mast import download_tess_lightcurve
from astromap.pipeline import preprocess
from astromap.inference import InferenceEngine
from astromap.vetting import perform_astrophysical_vetting
from astromap.reporting import write_report

print("AstroMap-AI environment loaded successfully.")

## 1. Real SPOC Photometric Ingestion from NASA MAST
We query the Mikulski Archive for Space Telescopes (MAST) for **TOI-700** (`TIC 150428135`), a multi-planet system hosting an Earth-sized habitable zone planet.

In [ ]:
tic_id = 150428135
observation = download_tess_lightcurve(tic_id)
print(f"Target: TIC {tic_id}")
print(f"Mission: {observation.mission} | Sector: {observation.sector}")
print(f"Total cadences: {observation.cadence_count}")
print(f"MAST Source URI: {observation.source_uri}")

## 2. Photometric Preprocessing & Dual-Domain CWT
We apply quality flag filtering, low-frequency trend removal using Savitzky-Golay filtering, median normalization, and generate the 2D Ricker Continuous Wavelet Transform scalogram.

In [ ]:
processed = preprocess(observation.curve)
print(f"Resampled cadences: {len(processed.time)}")
print(f"Valid points: {np.count_nonzero(processed.mask)}")
print(f"Scalogram dimensions: {processed.scalogram.shape} (scales x timestamps)")

## 3. Unified Hybrid Identification: PINN-TransNet + BLS Consensus
We run the full hybrid inference engine with the trained `pinn-transnet-v1.pt` checkpoint.

In [ ]:
checkpoint = root_dir / "artifacts" / "pinn-transnet-v1.pt"
engine = InferenceEngine(checkpoint=checkpoint if checkpoint.exists() else None)
prediction = engine.predict(tic_id)

print("=" * 60)
print(f"Detection Method:     {prediction.score_kind}")
print(f"Model Version:        {prediction.model_version}")
print(f"Unified Probability:  {prediction.probability * 100:.2f}%")
if prediction.ai_probability is not None:
    print(f"PINN-TransNet AI:     {prediction.ai_probability * 100:.2f}%")
if prediction.bls_score is not None:
    print(f"Classical BLS Score:  {prediction.bls_score * 100:.2f}%")
if prediction.harmonic_consistency is not None:
    print(f"Harmonic Consensus:   {prediction.harmonic_consistency * 100:.2f}%")
print(f"Estimated Period:     {prediction.period_days:.4f} days")
print(f"Estimated Depth:      {prediction.depth * 1e6:.1f} ppm")
print(f"Estimated Duration:   {prediction.duration_days * 24:.2f} hours")
print("=" * 60)

## 4. Automated Astrophysical Vetting Battery (Robovetter)
We verify the transit signal against false-positive scenarios:
- **Odd-Even Depth Parity Test** ($< 3.5\sigma$): rules out eclipsing binaries.
- **Secondary Eclipse Scan** (at phase $\phi = 0.5$): rules out self-luminous stars.
- **Stellar Density Consistency**: verifies physical transit density consistency against Gaia DR3.

In [ ]:
vetting = perform_astrophysical_vetting(
    curve=observation.curve,
    period_days=prediction.period_days,
    duration_days=prediction.duration_days,
    depth=prediction.depth,
    tic_id=tic_id,
)

print(f"Odd-Even Parity:       {'PASS' if vetting.odd_even_flag else 'FAIL'}")
print(f"Secondary Eclipse:     {'PASS' if vetting.secondary_eclipse_flag else 'FAIL'}")
print(f"Stellar Density (K-3): {'PASS' if vetting.stellar_density_flag else 'MARGINAL'}")
print(f"Final Disposition:     {vetting.final_disposition}")
for note in vetting.summary_notes:
    print(f"  - {note}")

## 5. Candidate Validation Report Generation
We export the self-contained HTML report ready for archiving or thesis submission.

In [ ]:
report_output = root_dir / "artifacts" / f"notebook_tic_{tic_id}_report.html"
write_report(prediction, report_output)
print(f"Report saved to: {report_output} ({report_output.stat().st_size} bytes)")